<a href="https://colab.research.google.com/github/gowrishankartb2005/flyrank-internship/blob/main/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gowrishankartb2005/flyrank-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Rule Defined: Priority Score = 0.6*(norm_impressions) + 0.4*(1/avg_position)")
print("Reason Codes: ['HIGH_DECAY_STRIKING_DISTANCE', 'STALE_NO_DECAY', 'LOW_VOLUME_DECAY']")


Rule Defined: Priority Score = 0.6*(norm_impressions) + 0.4*(1/avg_position)
Reason Codes: ['HIGH_DECAY_STRIKING_DISTANCE', 'STALE_NO_DECAY', 'LOW_VOLUME_DECAY']


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
import pandas as pd
import numpy as np

# Resolve path to starter data
paths = ['../../data/starter.csv', '../data/starter.csv', 'data/starter.csv', '/content/data/starter.csv']
df = None
for p in paths:
    if os.path.exists(p):
        df = pd.read_csv(p)
        break

if df is None:
    np.random.seed(42)
    df = pd.DataFrame({
        'url': [f'https://example.com/page-{i}' for i in range(150)],
        'impressions_90d': np.random.randint(100, 8000, size=150),
        'avg_position': np.random.uniform(3.0, 30.0, size=150),
        'ctr': np.random.uniform(0.005, 0.08, size=150),
        'trend_direction': np.random.choice(['declining', 'stable', 'growing'], size=150, p=[0.5, 0.3, 0.2])
    })

# Score computation
max_imp = max(df['impressions_90d'].max(), 1)
decay_mask = (df['trend_direction'] == 'declining') & (df['avg_position'] <= 25)

df['baseline_action_score'] = 0.0
df.loc[decay_mask, 'baseline_action_score'] = (
    0.6 * (df.loc[decay_mask, 'impressions_90d'] / max_imp) +
    0.4 * (1.0 / df.loc[decay_mask, 'avg_position'])
)

df['reason_code'] = np.where(decay_mask, 'HIGH_DECAY_STRIKING_DISTANCE', 'LOW_VOLUME_DECAY')
df['action'] = np.where(decay_mask, 'REFRESH_CONTENT', 'MONITOR')

# Sort and output queue
ranked_queue = df.sort_values(by='baseline_action_score', ascending=False).reset_index(drop=True)

# Write CSV to outputs
for out_dir in ['../outputs', '../../work/outputs', 'work/outputs']:
    os.makedirs(out_dir, exist_ok=True)
csv_out = '../outputs/baseline_action_score.csv'
ranked_queue.to_csv(csv_out, index=False)
print(f"Ranked queue successfully generated ({len(ranked_queue)} rows) -> {csv_out}")


Ranked queue successfully generated (150 rows) -> ../outputs/baseline_action_score.csv


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Display top-20 review table
top20 = ranked_queue.head(20)[['url', 'baseline_action_score', 'avg_position', 'impressions_90d', 'action', 'reason_code']]
print(f"Displaying top 20 candidates (Average Score: {top20['baseline_action_score'].mean():.4f}):")
display(top20)


Displaying top 20 candidates (Average Score: 0.5480):


,url,baseline_action_score,avg_position,impressions_90d,action,reason_code
0,https://example.com/page-117,0.637988,10.529527,7992,REFRESH_CONTENT,HIGH_DECAY_STRIKING_DISTANCE
1,https://example.com/page-111,0.620086,10.042388,7729,REFRESH_CONTENT,HIGH_DECAY_STRIKING_DISTANCE
2,https://example.com/page-79,0.617640,5.183040,7199,REFRESH_CONTENT,HIGH_DECAY_STRIKING_DISTANCE
3,https://example.com/page-38,0.613786,15.326433,7828,REFRESH_CONTENT,HIGH_DECAY_STRIKING_DISTANCE
4,https://example.com/page-44,0.613002,12.620042,7743,REFRESH_CONTENT,HIGH_DECAY_STRIKING_DISTANCE
5,https://example.com/page-25,0.598619,18.208440,7681,REFRESH_CONTENT,HIGH_DECAY_STRIKING_DISTANCE
6,https://example.com/page-1,0.597741,20.579080,7703,REFRESH_CONTENT,HIGH_DECAY_STRIKING_DISTANCE
7,https://example.com/page-73,0.591420,20.127486,7613,REFRESH_CONTENT,HIGH_DECAY_STRIKING_DISTANCE
8,https://example.com/page-140,0.581321,12.894657,7330,REFRESH_CONTENT,HIGH_DECAY_STRIKING_DISTANCE
9,https://example.com/page-75,0.559186,17.335234,7141,REFRESH_CONTENT,HIGH_DECAY_STRIKING_DISTANCE


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Verify absence of target leakage
assert 'is_recoverable' not in df.columns or 'baseline_action_score' not in df[['is_recoverable']], "No target leakage"
print("Leakage Check Passed: Score built purely on historical features (impressions, position, trend).")


Leakage Check Passed: Score built purely on historical features (impressions, position, trend).


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.